# Задание

Используя модуль **datasets** библиотеки **sklearn**, загрузите базу вин (`.load_wine()`).

Используя шаблон ноутбука, выполните загрузку, подготовку и предобработку данных. Обязательное условие: разделение данных на три выборки осуществляется по шаблону (изменять параметры подготовки данных запрещается)!

Проведите серию экспериментов и добейтесь максимальной точности классификации на тестовой выборке выше 94%.

---

С помощью метода `.summary()` зафиксируйте количество параметров созданной вами нейронной сети.


#Шаблон ноутбука

In [ ]:
# Последовательная модель НС
from tensorflow.keras.models import Sequential

# Основные слои
from tensorflow.keras.layers import Dense, Activation, Dropout, BatchNormalization

# Утилиты для to_categorical()
from tensorflow.keras import utils

# Алгоритмы оптимизации для обучения модели
from tensorflow.keras.optimizers import Adam

# Библиотека для работы с массивами
import numpy as np

# Отрисовка графиков
import matplotlib.pyplot as plt

# Разделение данных на выборки
from sklearn.model_selection import train_test_split

# Для загрузки датасета
from sklearn.datasets import load_wine

# Отрисовка изображений в ноутбуке, а не в консоли или файле
%matplotlib inline

##Описание базы

1. Датасет состоит из набора данных о винах и их классах.
2. Данные по одному вину хранятся в numpy-массиве `x_data`: (`13` параметров).
3. В датасете `3` класса вин: `y_data`.
4. Количество примеров: `178`.

In [ ]:
x_data = load_wine()['data']              # Загрузка набора данных о винах
y_data = load_wine()['target']            # Загрузка классов вин

print('Размерность x_data -', x_data.shape)
print('Размерность y_data -', y_data.shape)
print()

# Вывод примера данных
print('Данные по первому вину:',x_data[0])
print('Класс вина:',y_data[0])

Размерность x_data - (178, 13)
Размерность y_data - (178,)

Данные по первому вину: [1.423e+01 1.710e+00 2.430e+00 1.560e+01 1.270e+02 2.800e+00 3.060e+00
 2.800e-01 2.290e+00 5.640e+00 1.040e+00 3.920e+00 1.065e+03]
Класс вина: 0


##Подготовка данных

In [ ]:
# Перевод в one hot encoding
y_data = utils.to_categorical(y_data, 3)

# Разбиение наборов на общую и тестовую выборки
x_all, x_test, y_all, y_test = train_test_split(x_data,
                                                y_data,
                                                test_size=0.1,
                                                shuffle=True,
                                                random_state = 6)

# Разбиение общей выборки на обучающую и проверочную
x_train, x_val, y_train, y_val = train_test_split(x_all,
                                                  y_all,
                                                  test_size=0.1,
                                                  shuffle=True,
                                                  random_state = 6)

print(x_train.shape)
print(y_train.shape)
print()
print(x_val.shape)
print(y_val.shape)

(144, 13)
(144, 3)

(16, 13)
(16, 3)


In [ ]:
# ваше решение

# Импортируем tensorflow, если он еще не был импортирован как tf
import tensorflow as tf
# numpy и matplotlib.pyplot должны быть уже импортированы из предыдущих ячеек шаблона
# (import numpy as np, import matplotlib.pyplot as plt)
# Также предполагается, что tensorflow.keras.models.Sequential, 
# tensorflow.keras.layers.Dense, Dropout, BatchNormalization,
# tensorflow.keras.optimizers.Adam уже импортированы.

# Установка начальных состояний для воспроизводимости результатов
# Это важно для получения одинаковых результатов при каждом запуске кода.
tf.random.set_seed(42)
np.random.seed(42) # Для операций numpy, которые могут использоваться Keras неявно

# 1. Определение модели нейронной сети
# Мы будем использовать последовательную модель.
# Количество входных нейронов равно количеству признаков в x_train (x_train.shape[1]).
# Выходной слой будет иметь 3 нейрона (по одному для каждого класса вина) с активацией softmax.

model = Sequential(name="Wine_Classifier_NN")

# Первый скрытый слой с 64 нейронами и функцией активации ReLU.
# input_shape определяется количеством признаков (13 для датасета вин).
model.add(Dense(units=64, activation='relu', input_shape=(x_train.shape[1],)))
# Слой Dropout для регуляризации, помогает предотвратить переобучение.
# 25% нейронов будут случайно "отключены" на каждом шаге обучения.
model.add(Dropout(0.25))

# Второй скрытый слой с 32 нейронами и функцией активации ReLU.
model.add(Dense(units=32, activation='relu'))
# Еще один слой Dropout.
model.add(Dropout(0.25))

# Выходной слой. 3 нейрона (соответствуют 3 классам вин).
# Функция активации softmax используется для многоклассовой классификации,
# она преобразует выходы в вероятностное распределение по классам.
model.add(Dense(units=3, activation='softmax'))

# 2. Компиляция модели
# Оптимизатор Adam является хорошим выбором по умолчанию для многих задач.
# learning_rate=0.001 - стандартная скорость обучения.
# Функция потерь 'categorical_crossentropy' используется, так как метки классов представлены в формате one-hot encoding.
# Метрика 'accuracy' (точность) будет отслеживаться во время обучения и оценки.
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 3. Обучение модели
# epochs - количество полных проходов по всему набору обучающих данных.
# batch_size - количество обучающих примеров, используемых в одной итерации (обновлении весов).
# validation_data используется для оценки модели на проверочной выборке после каждой эпохи.
# verbose=0 отключает вывод логов обучения для каждой эпохи (можно поставить 1 или 2 для детального вывода).
epochs_count = 120  # Количество эпох может потребовать подстройки; начнем со 120
batch_size_count = 16 # Размер пакета

print("Начало обучения модели...")
history = model.fit(x_train, y_train,
                    epochs=epochs_count,
                    batch_size=batch_size_count,
                    validation_data=(x_val, y_val),
                    verbose=0) # verbose=0 для чистоты вывода в финальном решении
print("Обучение модели завершено.")

# 4. Оценка модели на тестовой выборке
# Это самый важный шаг для проверки, достигнута ли требуемая точность.
print("\n--- Оценка на тестовой выборке ---")
loss_test, accuracy_test = model.evaluate(x_test, y_test, verbose=0)

print(f"Потери на тестовой выборке: {loss_test:.4f}")
print(f"Точность на тестовой выборке: {accuracy_test:.4f} ({accuracy_test*100:.2f}%)")

# Проверка выполнения условия по точности
if accuracy_test > 0.94:
    print(f"УСПЕХ! Достигнута целевая точность >94% на тестовой выборке.")
    num_total_test = len(y_test)
    num_correct_test = int(round(accuracy_test * num_total_test))
    print(f"Это соответствует {num_correct_test} правильным классификациям из {num_total_test} тестовых примеров.")
    if num_correct_test < num_total_test -1: # Если больше 1 ошибки
         print(f"Примечание: достигнуто >94%, но количество ошибок ({num_total_test - num_correct_test}) может быть >1. На 18 тестовых образцах 17/18 = 94.44%.")

else:
    print(f"ВНИМАНИЕ: Целевая точность >94% на тестовой выборке НЕ достигнута.")
    print("Рекомендуется провести дополнительные эксперименты: ")
    print(" - Изменить архитектуру сети (количество слоев, нейронов, функции активации).")
    print(" - Подобрать параметры Dropout.")
    print(" - Изменить параметры оптимизатора (например, скорость обучения).")
    print(" - Увеличить/уменьшить количество эпох или размер пакета.")
    print(" - Попробовать BatchNormalization.")

# 5. Вывод информации о модели (структура и количество параметров)
# Это выполняет требование "зафиксируйте количество параметров".
print("\n--- Сводка по модели (Model Summary) ---")
model.summary()

# 6. Визуализация истории обучения
# Графики точности и потерь на обучающей и проверочной выборках.
print("\n--- Графики истории обучения ---")
plt.figure(figsize=(14, 6))

# График точности
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Точность на обучении (Train Accuracy)')
plt.plot(history.history['val_accuracy'], label='Точность на проверке (Validation Accuracy)')
plt.title('Динамика точности модели')
plt.xlabel('Эпохи')
plt.ylabel('Точность')
plt.legend()
plt.grid(True)

# График потерь
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Потери на обучении (Train Loss)')
plt.plot(history.history['val_loss'], label='Потери на проверке (Validation Loss)')
plt.title('Динамика потерь модели')
plt.xlabel('Эпохи')
plt.ylabel('Потери')
plt.legend()
plt.grid(True)

plt.tight_layout() # Автоматически корректирует параметры графика для плотного размещения
plt.show()